In [34]:
import pandas as pd
import numpy as np
import datetime as dt

#=========================================================================================
#DATA CLEANING PROCESS
#=========================================================================================

job_postings_fact= pd.read_csv("C:/temp/job_postings_fact.csv")
company_dim= pd.read_csv("C:/temp/company_dim.csv")
skills_job_dim= pd.read_csv("C:/temp/skills_job_dim.csv")
skills_dim= pd.read_csv("C:/temp/skills_dim.csv")

def clean_fact(df):
    df['job_posted_date']= pd.to_datetime(df['job_posted_date'])
    df['job_via']= df['job_via'].str.strip().str.replace('via', "")
    df['month_num']= df['job_posted_date'].dt.month
    df['month_name']= df['job_posted_date'].dt.strftime("%B")
    
    df['salary_adj']= np.where(df['salary_year_avg'].notna(), df['salary_year_avg'],
    np.where(df['salary_year_avg'].isna() & df['salary_hour_avg'].notna(),
     df['salary_hour_avg']*2080,np.nan))
    
    df= df.dropna(subset=['job_location','job_via','job_schedule_type','job_country',])
    df= df.drop(columns= ['job_title','search_location'])
    return(df)

def clean_company(df):
    df= df.drop(columns=[ 'link','link_google','thumbnail'])
    return(df)

def clean_schedule_type(df):
    df['job_schedule_type']= df['job_schedule_type'].astype(str)
    df['job_schedule_type']= df['job_schedule_type'].str.replace(r"[\[\]'\"']" , "", regex= True)
    df['job_schedule_type']= df['job_schedule_type'].str.replace(r"\s+and\s+", ",", regex= True, case= False)
    df['job_schedule_type']= df['job_schedule_type'].str.split(',')
    schedule_type= df[['job_id','company_id','job_schedule_type']].explode('job_schedule_type')
    return(schedule_type)


job_postings_fact= clean_fact(job_postings_fact)
schedule_type_dim=clean_schedule_type(job_postings_fact)
company_dim = clean_company(company_dim)

#Saving the cleaned datasets
job_postings_fact.to_csv("C:/Users/syabo/Documents/DATA ANALYSIS/Project Collection/Job_Market_Analysis_Project/datasets/job_postings_fact.csv", index= False)
company_dim.to_csv("C:/Users/syabo/Documents/DATA ANALYSIS/Project Collection/Job_Market_Analysis_Project/datasets/company_dim.csv", index= False)
skills_job_dim.to_csv("C:/Users/syabo/Documents/DATA ANALYSIS/Project Collection/Job_Market_Analysis_Project/datasets/skills_job_dim.csv", index= False)
skills_dim.to_csv("C:/Users/syabo/Documents/DATA ANALYSIS/Project Collection/Job_Market_Analysis_Project/datasets/skills_dim.csv", index= False)
schedule_type_dim.to_csv("C:/Users/syabo/Documents/DATA ANALYSIS/Project Collection/Job_Market_Analysis_Project/datasets/schedule_type_dim.csv", index= False)


